# Parker Solar Probe / FIELDS dynamic spectrum

This notebook plots the FIELDS Radio Frequency Spectrometer (RFS) data from
Parker Solar Probe (PSP) for a single day. The RFS is a digital receiver
covering two contiguous bands:

- **LFR** (Low Frequency Receiver): 10.5 kHz - 1.7 MHz
- **HFR** (High Frequency Receiver): 1.3 - 19.2 MHz

PSP/FIELDS Level-2 products are distributed as CDF files. The voltage power
spectral densities (PSDs) are converted to dB using a reference floor of
$10^{-16}\ \mathrm{V^2\ Hz^{-1}}$, following Pulupa et al. 2020
([10.3847/1538-4365/ab5dc0](https://doi.org/10.3847/1538-4365/ab5dc0)).

**Workflow:** load the LFR and HFR CDFs &rarr; convert PSD to dB &rarr; build
a single (time, frequency) DataFrame spanning both bands &rarr; remove the
per-channel median background &rarr; plot.

Two loaders are provided: a manual `pycdf` reader (default) and an alternative
based on `pyspedas`. Both result in the same `(time, freq, value_in_dB)`
quantities.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as ticker
from matplotlib.ticker import AutoMinorLocator
from matplotlib.colors import LogNorm
import matplotlib as mpl

# Use the precise matplotlib epoch (avoids ~10 us offsets in old matplotlib).
mpl.rcParams['date.epoch'] = '1970-01-01T00:00:00'
try:
    mdates.set_epoch('1970-01-01T00:00:00')
except RuntimeError:
    pass

# Unified plotting style for all dynamic spectra notebooks.
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['savefig.facecolor'] = 'white'
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['xtick.labelsize'] = 11
plt.rcParams['ytick.labelsize'] = 11


## Configuration


In [ ]:
# Date and I/O paths.
mydate = '2019-04-01'
year, month, day = mydate.split('-')

data_dir = './sample_data/psp'
outputs  = './outputs'
os.makedirs(outputs, exist_ok=True)

# PSD floor in V^2/Hz (used for dB conversion, see Pulupa et al. 2020).
PSD_FLOOR = 1e-16


## Helper functions


In [ ]:
def subtract_background_median(df):
    """
    Subtract a per-channel median background from a dynamic spectrum.

    The function computes the median along the time axis (axis=0) for each
    frequency channel, then subtracts it from every time sample of that
    channel. This is the standard approach for highlighting transient
    emission against a slowly-varying instrumental/sky background.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame with shape (n_times, n_freqs). Index is time, columns
        are frequencies.

    Returns
    -------
    pandas.DataFrame
        Same shape as input with the per-channel median removed.
    """
    bkg = np.nanmedian(df.values, axis=0)
    return df - np.tile(bkg, (df.shape[0], 1))


def psd_to_db(arr, floor=PSD_FLOOR):
    """
    Convert linear PSD (V^2/Hz) to decibels above a fixed floor.

    Parameters
    ----------
    arr : array-like
        Linear PSD values.
    floor : float
        Reference PSD floor in the same units as `arr`.

    Returns
    -------
    numpy.ndarray
        PSD in dB.
    """
    return 10.0 * np.log10(np.asarray(arr) / floor)


## Load PSP/RFS data with `pycdf` (default)

If you prefer `pyspedas`, skip to the next section.


In [ ]:
# spacepy.pycdf needs the NASA CDF library; pip install spacepy and
# install the CDF library separately (or use cdflib instead, see below).
from spacepy import pycdf

lfr_path = f'{data_dir}/psp_fld_l2_rfs_lfr_{year}{month}{day}_v02.cdf'
hfr_path = f'{data_dir}/psp_fld_l2_rfs_hfr_{year}{month}{day}_v02.cdf'

cdf_lfr = pycdf.CDF(lfr_path)
cdf_hfr = pycdf.CDF(hfr_path)

# LFR: time, frequency, PSD.
tm_lfr   = np.asarray(cdf_lfr['epoch_lfr'])
freq_lfr = np.asarray(cdf_lfr['frequency_lfr_auto_averages_ch0_V1V2'])[1] / 1e6  # MHz
lp_lfr   = psd_to_db(cdf_lfr['psp_fld_l2_rfs_lfr_auto_averages_ch0_V1V2'])

# HFR: time, frequency, PSD.
tm_hfr   = np.asarray(cdf_hfr['epoch_hfr'])
freq_hfr = np.asarray(cdf_hfr['frequency_hfr_auto_averages_ch0_V1V2'])[1] / 1e6  # MHz
lp_hfr   = psd_to_db(cdf_hfr['psp_fld_l2_rfs_hfr_auto_averages_ch0_V1V2'])

print(f'LFR shape (time, freq): {lp_lfr.shape}, freq range {freq_lfr[0]*1e3:.1f} kHz - {freq_lfr[-1]:.2f} MHz')
print(f'HFR shape (time, freq): {lp_hfr.shape}, freq range {freq_hfr[0]:.2f} - {freq_hfr[-1]:.2f} MHz')


## Alternative loader: `pyspedas`

`pyspedas.psp.fields` downloads and loads PSP/FIELDS data via tplot. Use this
section instead of the pycdf block above if preferred (and comment out the
pycdf cell).


In [ ]:
# Uncomment to use pyspedas as an alternative loader.
#
# import pyspedas
# from pytplot import get_data
#
# pyspedas.psp.fields(trange=[mydate, mydate], datatype='rfs_lfr', level='l2', no_update=True)
# pyspedas.psp.fields(trange=[mydate, mydate], datatype='rfs_hfr', level='l2', no_update=True)
#
# lfr_var = 'psp_fld_l2_rfs_lfr_auto_averages_ch0_V1V2'
# hfr_var = 'psp_fld_l2_rfs_hfr_auto_averages_ch0_V1V2'
#
# tm_lfr_arr, lp_lfr_lin, freq_lfr_hz = get_data(lfr_var)
# tm_hfr_arr, lp_hfr_lin, freq_hfr_hz = get_data(hfr_var)
#
# tm_lfr   = pd.to_datetime(tm_lfr_arr, unit='s').to_pydatetime()
# tm_hfr   = pd.to_datetime(tm_hfr_arr, unit='s').to_pydatetime()
# freq_lfr = freq_lfr_hz / 1e6
# freq_hfr = freq_hfr_hz / 1e6
# lp_lfr   = psd_to_db(lp_lfr_lin)
# lp_hfr   = psd_to_db(lp_hfr_lin)


## Concatenate LFR and HFR into a single DataFrame


In [ ]:
# Build per-band DataFrames indexed by frequency, columns are timestamps.
df_lfr = pd.DataFrame(lp_lfr.T, index=freq_lfr, columns=tm_lfr)
df_hfr = pd.DataFrame(lp_hfr.T, index=freq_hfr, columns=tm_hfr[:lp_hfr.shape[0]])

# Reindex both onto a common time axis (LFR is the slower band) and stack.
common_times = tm_lfr
df_lfr = df_lfr.reindex(columns=common_times, method='nearest')
df_hfr = df_hfr.reindex(columns=common_times, method='nearest')

df_psp = pd.concat([df_lfr, df_hfr]).sort_index()
# Keep the first occurrence in the overlap region (around 1.3-1.7 MHz).
df_psp = df_psp[~df_psp.index.duplicated(keep='first')]

print(f'Combined PSP shape (freq, time): {df_psp.shape}')
print(f'Freq range: {df_psp.index.min()*1e3:.1f} kHz - {df_psp.index.max():.2f} MHz')


## Remove the per-channel median background


In [ ]:
# subtract_background_median expects (time, freq), so transpose first.
df_psp_nobkg = subtract_background_median(df_psp.T).T


## Plot the combined dynamic spectrum


In [ ]:
vmin = np.nanpercentile(df_psp_nobkg.values, 50)
vmax = np.nanpercentile(df_psp_nobkg.values, 99)

fig = plt.figure(figsize=(11, 6))
ax  = fig.add_subplot(111)
pc  = ax.pcolormesh(
    df_psp_nobkg.columns, df_psp_nobkg.index, df_psp_nobkg.values,
    vmin=vmin, vmax=vmax, cmap='Spectral_r',
)
fig.colorbar(pc, ax=ax, pad=0.02, label='dB above 1e-16 V$^2$/Hz')

ax.set_yscale('log')
ax.set_ylim(ax.get_ylim()[::-1])  # high frequency at top
ax.set_xlabel('Time (UT)')
ax.set_ylabel('Frequency (MHz)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.set_title(f'PSP / FIELDS RFS  -  {mydate}')

fig.tight_layout()
fig.savefig(f'{outputs}/psp_dyspec_{mydate}.png', bbox_inches='tight')
plt.show()
